# 🎵 Audio Decomposition with SVD: Hearing the Mathematics of Sound

---

**Author:** ¡Aaron Smolyar!

**Course:** Linear Algebra / PCA Applications

**Date:** December 2025

---

## Overview

This notebook demonstrates a beautiful application of **Singular Value Decomposition (SVD)** to real audio data. We'll decompose sound into its fundamental "audio atoms" and explore how much information we can throw away while still preserving the essence of what we hear.

### The Key Insight

A **spectrogram** is simply a matrix $M \in \mathbb{R}^{F \times T}$ where:
- Rows = frequency bins (what pitches are present)
- Columns = time frames (when things happen)
- Values = magnitude (how loud each frequency is at each moment)

SVD gives us:
$$M \approx \sum_{i=1}^{k} \sigma_i \, u_i \, v_i^\top$$

Each term is a **rank-1 pattern**:
- $u_i$ = "frequency profile" (what frequencies are involved)
- $v_i$ = "time activity" (when it happens)  
- $\sigma_i$ = importance weight

As we increase $k$, we're literally adding more "audio atoms" back into our reconstruction!

---


In [ ]:
# Install required packages (silent mode)
!pip -q install librosa soundfile

# Core imports
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Audio, display, Markdown, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, Layout
import warnings
warnings.filterwarnings('ignore')
import gc

# Set up beautiful plot styling
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#16213e'
plt.rcParams['axes.edgecolor'] = '#e94560'
plt.rcParams['axes.labelcolor'] = '#eaeaea'
plt.rcParams['text.color'] = '#eaeaea'
plt.rcParams['xtick.color'] = '#eaeaea'
plt.rcParams['ytick.color'] = '#eaeaea'
plt.rcParams['grid.color'] = '#0f3460'
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.dpi'] = 100

# Custom colormap for spectrograms
colors = ['#0f0f23', '#1a1a2e', '#16213e', '#0f3460', '#e94560', '#ff6b6b', '#feca57', '#ffffff']
custom_cmap = LinearSegmentedColormap.from_list('audio_heat', colors, N=256)

print(":) All dependencies loaded successfully!")
print(f":O NumPy version: {np.__version__}")
print(f";) Librosa version: {librosa.__version__}")


---

## 2. Load Your Audio

You have **three options** to get audio into this notebook:

### Option A: 🎵 Use the pre-loaded song (recommended!)
**"Running Through Me" by Tom Misch** (NPR Music Tiny Desk Concert Version) — my favorite song of all time! Ready to download analyze. (skip the next cell)

### Option B: Upload a file
Upload your own audio file (WAV, MP3, FLAC, etc.) (skip the next cell)

### Option C: Record directly  
Use your microphone to record audio right in the browser! (use the next cell, skip the one that follows)

**Tips for best results:**
- Speech and music both work great, but show different compression characteristics
- The audio is converted to mono and normalized automatically

**IF YOU ARE HAVING PERMISSIONS ISSUES WITH RECORDING:**
- Please try to rerun the cell, that has worked for me.
- You can also try to go into your Chrome settings for the site near the navbar and manually allow. The effort will be worth it!! If you can't, then just upload an audio file manually, or use TOM MISCH THE GOAT in the section after next.


In [ ]:
# Audio Recording functionality using JavaScript
# This creates an in-browser recorder that saves to a WAV file

from IPython.display import HTML, display, Javascript
from google.colab import output
import base64
import io

# JavaScript code for audio recording
RECORD_JS = """
<style>
  .audio-controls {
    font-family: 'Courier New', monospace;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    padding: 20px;
    border-radius: 12px;
    border: 2px solid #e94560;
    margin: 10px 0;
    text-align: center;
  }
  .audio-controls h3 {
    color: #e94560;
    margin-bottom: 15px;
  }
  .record-btn {
    background: linear-gradient(135deg, #e94560 0%, #ff6b6b 100%);
    color: white;
    border: none;
    padding: 12px 30px;
    font-size: 16px;
    border-radius: 25px;
    cursor: pointer;
    margin: 5px;
    transition: all 0.3s ease;
    font-family: 'Courier New', monospace;
  }
  .record-btn:hover {
    transform: scale(1.05);
    box-shadow: 0 0 20px rgba(233, 69, 96, 0.5);
  }
  .record-btn:disabled {
    background: #555;
    cursor: not-allowed;
    transform: none;
    box-shadow: none;
  }
  .record-btn.recording {
    background: linear-gradient(135deg, #ff0000 0%, #ff6b6b 100%);
    animation: pulse 1s infinite;
  }
  @keyframes pulse {
    0%, 100% { box-shadow: 0 0 10px rgba(255, 0, 0, 0.5); }
    50% { box-shadow: 0 0 25px rgba(255, 0, 0, 0.8); }
  }
  .status-text {
    color: #feca57;
    margin-top: 10px;
    font-size: 14px;
  }
  .timer {
    color: #ffffff;
    font-size: 24px;
    margin: 10px 0;
  }
</style>

<div class="audio-controls">
  <h3>🎙️ Audio Recorder</h3>
  <div class="timer" id="timer">00:00</div>
  <button id="recordBtn" class="record-btn" onclick="toggleRecording()">🔴 Start Recording</button>
  <button id="stopBtn" class="record-btn" onclick="stopRecording()" disabled>⏹️ Stop & Save</button>
  <div class="status-text" id="status">Click "Start Recording" to begin</div>
</div>

<script>
let mediaRecorder;
let audioChunks = [];
let startTime;
let timerInterval;

async function toggleRecording() {
  const recordBtn = document.getElementById('recordBtn');
  const stopBtn = document.getElementById('stopBtn');
  const status = document.getElementById('status');

  if (!mediaRecorder || mediaRecorder.state === 'inactive') {
    try {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      mediaRecorder = new MediaRecorder(stream);
      audioChunks = [];

      mediaRecorder.ondataavailable = (event) => {
        audioChunks.push(event.data);
      };

      mediaRecorder.onstop = async () => {
        const audioBlob = new Blob(audioChunks, { type: 'audio/wav' });
        const reader = new FileReader();
        reader.onloadend = () => {
          const base64data = reader.result.split(',')[1];
          google.colab.kernel.invokeFunction('notebook.save_recording', [base64data], {});
        };
        reader.readAsDataURL(audioBlob);

        // Stop all tracks
        stream.getTracks().forEach(track => track.stop());
        status.textContent = '✅ Recording saved! Processing...';
      };

      mediaRecorder.start();
      startTime = Date.now();
      timerInterval = setInterval(updateTimer, 100);

      recordBtn.textContent = '⏸️ Pause';
      recordBtn.classList.add('recording');
      stopBtn.disabled = false;
      status.textContent = '🔴 Recording... Speak now!';

    } catch (err) {
      status.textContent = '❌ Error: ' + err.message + '. Make sure to allow microphone access!';
    }
  } else if (mediaRecorder.state === 'recording') {
    mediaRecorder.pause();
    clearInterval(timerInterval);
    recordBtn.textContent = '▶️ Resume';
    recordBtn.classList.remove('recording');
    status.textContent = '⏸️ Paused';
  } else if (mediaRecorder.state === 'paused') {
    mediaRecorder.resume();
    timerInterval = setInterval(updateTimer, 100);
    recordBtn.textContent = '⏸️ Pause';
    recordBtn.classList.add('recording');
    status.textContent = '🔴 Recording...';
  }
}

function stopRecording() {
  if (mediaRecorder && mediaRecorder.state !== 'inactive') {
    mediaRecorder.stop();
    clearInterval(timerInterval);
    document.getElementById('recordBtn').textContent = '🔴 Start Recording';
    document.getElementById('recordBtn').classList.remove('recording');
    document.getElementById('stopBtn').disabled = true;
  }
}

function updateTimer() {
  const elapsed = Date.now() - startTime;
  const seconds = Math.floor(elapsed / 1000);
  const minutes = Math.floor(seconds / 60);
  const secs = seconds % 60;
  document.getElementById('timer').textContent =
    String(minutes).padStart(2, '0') + ':' + String(secs).padStart(2, '0');
}
</script>
"""

# Global variable to store recorded audio
recorded_audio_data = None

def save_recording(base64_audio):
    """Callback function to save recorded audio from JavaScript."""
    global recorded_audio_data
    recorded_audio_data = base64_audio
    print("✅ Recording captured! Run the next cell to process it.")

# Register the callback
output.register_callback('notebook.save_recording', save_recording)

# Display the recorder
display(HTML(RECORD_JS))

print("\n" + "="*60)
print("  OR upload a file instead in the next section:")
print("="*60)



### Choose Your Audio Source

**Run the next cell to select your audio:**
- 🎵 **Option 1**: Use "Running Through Me" by Tom Misch (the best song ever!)
- 📁 **Option 2**: Upload your own file
- 🎙️ **Option 3**: Skip if you already recorded above

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHOOSE YOUR AUDIO SOURCE
# ═══════════════════════════════════════════════════════════════════════════
# Set this to True to use Running Through Me by Tom Misch (you will not regret),
# or False to upload your own file!

USE_TOM_MISCH = True  # 🎵 "Running Through Me"

# ═══════════════════════════════════════════════════════════════════════════

from google.colab import files
import os

if USE_TOM_MISCH:
    # Download Tom Misch - Running Through Me from Google Drive
    print("🎵 Downloading 'Running Through Me' by Tom Misch...")

    # Install gdown for Google Drive downloads
    !pip -q install gdown
    import gdown

    # Google Drive file ID from the sharing link
    file_id = "1TXiqbe8YBNXQVDJsCwBCjqSzNdZPIHz3"
    audio_path = "TomMisch_RunningThroughMe.mp3"

    # Download if not already present
    if not os.path.exists(audio_path):
        gdown.download(f"https://drive.google.com/uc?id={file_id}", audio_path, quiet=False)

    audio_source = 'tom_misch'
    print(f"\n✅ Ready: {audio_path}")
else:
    # Option to upload a file
    print("📁 Upload an audio file (or skip if you recorded above)")
    print("   Supported formats: WAV, MP3, FLAC, OGG, etc.\n")

    try:
        uploaded = files.upload()
        if uploaded:
            audio_path = next(iter(uploaded.keys()))
            print(f"\n✅ Uploaded: {audio_path}")
            audio_source = 'upload'
        else:
            audio_source = None
    except:
        audio_source = None
        print("   (No file uploaded - will use recording if available)")


Now, we can process the audio!

In [ ]:
# Process audio from recording, upload, or pre-loaded song
import soundfile as sf

# ═══════════════════════════════════════════════════════════════════════════
# AUDIO PROCESSING SETTINGS
# ═══════════════════════════════════════════════════════════════════════════
TARGET_SR = 22050  # Sample rate (22kHz is sufficient for speech and most music)
MAX_DURATION = 90  # Duration in seconds (set to None for full audio)
USE_FLOAT32 = True  # Use float32 precision
# ═══════════════════════════════════════════════════════════════════════════

# Determine which audio source to use
if 'audio_source' in dir() and audio_source == 'tom_misch':
    # Load Tom Misch - Running Through Me (my favorite song!)
    print("🎵 Loading 'Running Through Me' by Tom Misch...")
    # Use duration parameter directly in load for efficiency
    y_original, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True, duration=MAX_DURATION)
    source_name = "Tom Misch - Running Through Me"

elif 'audio_source' in dir() and audio_source == 'upload':
    # Load from uploaded file
    print("📂 Loading from uploaded file...")
    y_original, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True, duration=MAX_DURATION)
    source_name = audio_path

elif 'recorded_audio_data' in dir() and recorded_audio_data is not None:
    # Load from recording
    print("🎙️ Loading from recording...")
    audio_bytes = base64.b64decode(recorded_audio_data)

    # Save temporarily and load with librosa for consistent processing
    temp_path = '/tmp/recorded_audio.wav'
    with open(temp_path, 'wb') as f:
        f.write(audio_bytes)

    y_original, sr = librosa.load(temp_path, sr=TARGET_SR, mono=True, duration=MAX_DURATION)
    source_name = "Recorded Audio"

else:
    raise ValueError("❌ No audio source found! Please either:\n"
                     "   1. Set USE_TOM_MISCH = True to use Running Through Me, OR\n"
                     "   2. Upload an audio file, OR\n"
                     "   3. Record audio using the recorder above\n"
                     "   Then run this cell again.")

# Normalize to [-1, 1] and convert to float32
y = y_original / (np.max(np.abs(y_original)) + 1e-9)
if USE_FLOAT32:
    y = y.astype(np.float32)

del y_original
gc.collect()

# Calculate duration
duration = len(y) / sr

print(f"\n🎵 Audio Properties:")
print(f"   • Source: {source_name}")
print(f"   • Sample rate: {sr} Hz")
print(f"   • Duration: {duration:.2f} seconds")
print(f"   • Total samples: {len(y):,}")
print(f"   • Data type: {y.dtype}")
print(f"\n▶️ Original Audio:")
display(Audio(y, rate=sr))


---

## 3. Short-Time Fourier Transform (STFT)

The STFT converts our 1D audio signal into a 2D **time-frequency representation**.

### How it works:
1. Slide a window across the audio signal
2. At each position, compute the FFT to get frequency content
3. Stack these snapshots to form a matrix

**Parameters:**
- `n_fft = 2048`: Window size (higher = better frequency resolution, worse time resolution)
- `hop_length = 512`: Step size between windows (overlap = n_fft - hop_length)

The output $X \in \mathbb{C}^{F \times T}$ is complex-valued, containing both **magnitude** and **phase** information.


In [ ]:
# STFT parameters
n_fft = 1024      # FFT window size
hop_length = 256  # Hop between windows (~11ms resolution)

# Compute STFT (complex-valued)
print("Computing STFT...", end=" ")
X = librosa.stft(y, n_fft=n_fft, hop_length=hop_length).astype(np.complex64)

# Separate magnitude and phase
M = np.abs(X).astype(np.float32)       # Magnitude spectrogram (what we'll decompose with SVD)
P = np.angle(X).astype(np.float32)     # Phase (we'll reuse this for reconstruction)

del X
gc.collect()
print("Done!")

print(f"\n📊 Spectrogram Matrix M:")
print(f"   • Shape: {M.shape[0]} frequency bins × {M.shape[1]} time frames")
print(f"   • Total elements: {M.shape[0] * M.shape[1]:,}")
print(f"   • Data size: {M.shape[0] * M.shape[1]:,} elements")
print(f"   • Frequency resolution: {sr/n_fft:.1f} Hz per bin")
print(f"   • Time resolution: {hop_length/sr*1000:.1f} ms per frame")
print(f"\n   This is the matrix we'll decompose using SVD!")


In [ ]:
# Visualize the original spectrogram
fig, ax = plt.subplots(figsize=(14, 5))

# Convert to dB scale for better visualization (compute inline, don't store)
img = librosa.display.specshow(
    librosa.amplitude_to_db(M, ref=np.max),
    sr=sr,
    hop_length=hop_length,
    x_axis='time',
    y_axis='hz',
    cmap=custom_cmap,
    ax=ax
)

ax.set_title('Original Spectrogram (dB Scale)', fontsize=14, fontweight='bold', color='#e94560')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Frequency (Hz)', fontsize=11)

cbar = fig.colorbar(img, ax=ax, format='%+2.0f dB')
cbar.ax.yaxis.set_tick_params(color='#eaeaea')
cbar.outline.set_edgecolor('#e94560')

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print("\n💡 This spectrogram shows HOW LOUD each FREQUENCY is at each moment in TIME.")
print("   Bright regions = loud frequencies. Dark regions = quiet/absent frequencies.")


---

## 4. Singular Value Decomposition (SVD)

Now for the main event! We decompose our magnitude spectrogram $M$ using SVD:

$$M = U \Sigma V^\top$$

Where:
- $U \in \mathbb{R}^{F \times r}$ — Left singular vectors (frequency patterns)
- $\Sigma \in \mathbb{R}^{r \times r}$ — Diagonal matrix of singular values (importance weights)
- $V^\top \in \mathbb{R}^{r \times T}$ — Right singular vectors (time patterns)
- $r = \min(F, T)$ is the rank

### The Magic of Truncation

By keeping only the top $k$ singular values, we get a **low-rank approximation**:

$$M_k = U_k \Sigma_k V_k^\top \approx M$$

The Eckart-Young theorem guarantees this is the **best rank-k approximation** in the Frobenius norm!


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRUNCATED SVD
# ═══════════════════════════════════════════════════════════════════════════
# We only need the top K components to capture most of the signal energy.
# ═══════════════════════════════════════════════════════════════════════════

from scipy.sparse.linalg import svds

K_MAX = 100  # Top components to compute (captures 99%+ energy for audio)

print(f"🔢 Computing truncated SVD (top {K_MAX} of {min(M.shape)} components)...")

# svds requires float64 internally
M_f64 = M.astype(np.float64)
U, S, Vt = svds(M_f64, k=K_MAX)
del M_f64
gc.collect()

# svds returns singular values in ascending order - sort descending
idx = np.argsort(S)[::-1]
S = S[idx].astype(np.float32)
U = U[:, idx].astype(np.float32)
Vt = Vt[idx, :].astype(np.float32)

print("Done!\n")

print(f"📐 SVD Components (top {K_MAX}):")
print(f"   • U  (frequency patterns): {U.shape}")
print(f"   • Σ  (singular values):    {S.shape}")
print(f"   • Vᵀ (time patterns):      {Vt.shape}")
print(f"\n   Largest σ₁ = {S[0]:.2f}")
print(f"   Smallest σ_{K_MAX} = {S[-1]:.4f}")

gc.collect()


---

## 5. Energy Analysis: How Much Information is in Each Component?

The **energy** captured by each singular value is proportional to $\sigma_i^2$.

The cumulative energy tells us: "If we keep the top $k$ components, how much of the total information do we retain?"

$$\text{Energy}_k = \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2}$$

This is a key insight for **compression**: we can often capture 90%+ of the energy with just a fraction of the components!


In [ ]:
# Calculate energy distribution (based on truncated SVD)
# Note: Energy percentages are relative to the top K_MAX components
# For audio, the top 100 components typically capture 99.9%+ of total energy
energy = (S ** 2) / np.sum(S ** 2)
cum_energy = np.cumsum(energy)

# Find k for different energy thresholds (within our truncated range)
k_90 = np.searchsorted(cum_energy, 0.90) + 1
k_95 = np.searchsorted(cum_energy, 0.95) + 1
k_99 = min(np.searchsorted(cum_energy, 0.99) + 1, len(S))  # Cap at available components

print(f"📈 Energy Thresholds (within top {len(S)} components):")
print(f"   • 90% energy: k = {k_90} components ({k_90/len(S)*100:.1f}% of computed)")
print(f"   • 95% energy: k = {k_95} components ({k_95/len(S)*100:.1f}% of computed)")
print(f"   • 99% energy: k = {k_99} components ({k_99/len(S)*100:.1f}% of computed)")
print(f"\n   Using top {len(S)} components of the SVD.")


In [ ]:
# Visualize energy distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Singular values (log scale)
ax1 = axes[0]
ax1.semilogy(S, color='#e94560', linewidth=2, label='Singular values σᵢ')
ax1.axhline(y=S[k_90-1], color='#feca57', linestyle='--', alpha=0.7, label=f'90% threshold (k={k_90})')
ax1.axhline(y=S[k_95-1], color='#ff6b6b', linestyle='--', alpha=0.7, label=f'95% threshold (k={k_95})')
ax1.set_xlabel('Component Index', fontsize=11)
ax1.set_ylabel('Singular Value (log scale)', fontsize=11)
ax1.set_title('Singular Value Spectrum', fontsize=14, fontweight='bold', color='#e94560')
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, len(S)])

# Right plot: Cumulative energy
ax2 = axes[1]
ax2.plot(cum_energy * 100, color='#e94560', linewidth=2)
ax2.axhline(y=90, color='#feca57', linestyle='--', alpha=0.7)
ax2.axhline(y=95, color='#ff6b6b', linestyle='--', alpha=0.7)
ax2.axhline(y=99, color='#ffffff', linestyle='--', alpha=0.5)

# Mark the threshold points
ax2.scatter([k_90], [90], color='#feca57', s=100, zorder=5, edgecolor='white')
ax2.scatter([k_95], [95], color='#ff6b6b', s=100, zorder=5, edgecolor='white')
ax2.scatter([k_99], [99], color='#ffffff', s=100, zorder=5, edgecolor='white')

ax2.annotate(f'k={k_90}', (k_90, 90), textcoords="offset points", xytext=(10,-15), color='#feca57', fontsize=10)
ax2.annotate(f'k={k_95}', (k_95, 95), textcoords="offset points", xytext=(10,-15), color='#ff6b6b', fontsize=10)
ax2.annotate(f'k={k_99}', (k_99, 99), textcoords="offset points", xytext=(10,5), color='#ffffff', fontsize=10)

ax2.set_xlabel('Number of Components (k)', fontsize=11)
ax2.set_ylabel('Cumulative Energy (%)', fontsize=11)
ax2.set_title('Cumulative Energy Captured', fontsize=14, fontweight='bold', color='#e94560')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, len(S)])
ax2.set_ylim([0, 102])

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print("\n💡 Notice how quickly the cumulative energy rises! This shows that most of the")
print("   'information' in the audio is concentrated in just a few dominant patterns.")


---

## 6. Visualizing the Top Singular Components

Let's look at what the SVD actually found! Each component $(u_i, v_i)$ represents a **time-frequency pattern**:

- **$u_i$** (left singular vector): The **frequency profile** — which frequencies participate in this pattern
- **$v_i$** (right singular vector): The **time activity** — when this pattern is active

The outer product $\sigma_i \cdot u_i \cdot v_i^\top$ gives the rank-1 contribution of this "audio atom" to the full spectrogram.

### **Note: The following cell is probably the slowest in the entire notebook. Have patience!**


In [ ]:
# Visualize top 3 components
n_components_to_show = 3
fig, axes = plt.subplots(n_components_to_show, 2, figsize=(12, 2.5*n_components_to_show))

# Frequency axis for plotting
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

for i in range(n_components_to_show):
    # Left: Frequency profile (uᵢ) - very lightweight, just plotting a vector
    ax1 = axes[i, 0]
    ax1.plot(freqs, np.abs(U[:, i]), color='#e94560', linewidth=1.5)
    ax1.fill_between(freqs, np.abs(U[:, i]), alpha=0.3, color='#e94560')
    ax1.set_xlabel('Frequency (Hz)' if i == n_components_to_show-1 else '')
    ax1.set_ylabel(f'Comp {i+1}', fontsize=10)
    ax1.set_title(f'Frequency Profile (uᵢ)' if i == 0 else '', fontsize=11, fontweight='bold', color='#e94560')
    ax1.set_xlim([0, sr/2])
    ax1.grid(True, alpha=0.3)

    # Right: Rank-1 spectrogram - compute dB inline to avoid storing intermediate
    ax2 = axes[i, 1]
    rank1_db = librosa.amplitude_to_db(
        np.abs(S[i] * np.outer(U[:, i], Vt[i, :])),  # Computed inline
        ref=np.max
    )
    librosa.display.specshow(
        rank1_db,
        sr=sr, hop_length=hop_length,
        x_axis='time' if i == n_components_to_show-1 else None,
        y_axis='hz',
        cmap=custom_cmap, ax=ax2
    )
    del rank1_db
    gc.collect()
    
    energy_pct = energy[i] * 100
    ax2.set_title(f'Rank-1 Pattern (σ={S[i]:.0f}, {energy_pct:.1f}% energy)' if i == 0 else f'σ={S[i]:.0f}, {energy_pct:.1f}%',
                  fontsize=10, color='#feca57')
    ax2.set_ylabel('')

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print("\n💡 Each row: frequency profile × time pattern = one 'audio atom'.")


### What Are These Components? (Not Just Single Frequencies!)

Each singular component is **NOT** a single frequency — it's something more powerful!

**The Math:**
$$\text{Component}_i = \sigma_i \cdot u_i \cdot v_i^\top$$

| Part | What It Is | Interpretation |
|------|-----------|----------------|
| $u_i$ | Left singular vector (F×1) | **Frequency profile** — which frequencies participate, and how much |
| $v_i$ | Right singular vector (1×T) | **Time activity** — when this pattern is "on" |
| $\sigma_i$ | Singular value (scalar) | **Importance weight** |

**Key Insight:** SVD finds **patterns that co-occur**. For example:
- A musical note with its harmonics (they always sound together → one component)
- A vowel sound with multiple formant frequencies
- A drum hit that has many frequencies but a specific time shape

**Below you'll hear two versions of each component:**
1. **Time-varying**: The actual rank-1 reconstruction (frequency profile × time activity)
2. **Steady tone**: Just the frequency profile played at constant volume for 3 seconds — so you can hear *what frequencies* are in that pattern


In [ ]:
# Listen to each singular component individually!
# Listen to individual singular components

print("🎧 Listen to Individual Singular Components")
print("="*60)
print("For each component, you'll hear:")
print("   • Steady tone: The frequency profile as a sustained 2-second sound")
print("   • Time-varying: The reconstruction with time dynamics from vᵢ\n")

n_components_to_play = 3
steady_duration = 2.0  # seconds for the steady tone preview

for i in range(n_components_to_play):
    energy_pct = energy[i] * 100
    print(f"{'─'*60}")
    print(f"🎵 COMPONENT {i+1}: σ = {S[i]:.2f} ({energy_pct:.2f}% of total energy)")
    print(f"{'─'*60}")

    # ═══════════════════════════════════════════════════════════════
    # VERSION 1: Steady tone (frequency profile held constant for 2 sec)
    # ═══════════════════════════════════════════════════════════════
    n_frames_steady = int(steady_duration * sr / hop_length)
    freq_profile = (np.abs(U[:, i]) * S[i]).astype(np.float32)
    steady_magnitude = np.tile(freq_profile.reshape(-1, 1), (1, n_frames_steady))
    random_phase = np.random.uniform(-np.pi, np.pi, steady_magnitude.shape).astype(np.float32)
    steady_complex = steady_magnitude * np.exp(1j * random_phase)
    y_steady = librosa.istft(steady_complex, hop_length=hop_length)
    y_steady = y_steady / (np.max(np.abs(y_steady)) + 1e-9) * 0.8
    del steady_magnitude, random_phase, steady_complex
    
    print(f"\n▶️ Steady tone (2 sec) — frequency profile u_{i+1}:")
    display(Audio(y_steady, rate=sr))
    del y_steady
    gc.collect()

    # ═══════════════════════════════════════════════════════════════
    # VERSION 2: Time-varying (actual rank-1 reconstruction)
    # ═══════════════════════════════════════════════════════════════
    rank1_magnitude = (S[i] * np.outer(U[:, i], Vt[i, :])).astype(np.float32)
    rank1_complex = rank1_magnitude * np.exp(1j * P)
    del rank1_magnitude
    y_timevarying = librosa.istft(rank1_complex, hop_length=hop_length, length=len(y))
    del rank1_complex
    y_timevarying = y_timevarying / (np.max(np.abs(y_timevarying)) + 1e-9)

    print(f"▶️ Time-varying (with v_{i+1} dynamics):")
    display(Audio(y_timevarying, rate=sr))
    print()
    
    del y_timevarying
    gc.collect()

print("💡 The steady tones reveal WHAT FREQUENCIES each component contains.")
print("   The time-varying versions show HOW those frequencies evolve over time.")


---

## 7. Interactive Reconstruction Explorer

Now for the fun part! Use the slider to select a value of k, then **click the button** to reconstruct and hear the audio.

### Reconstruction Pipeline:
1. **Truncate SVD**: $M_k = U_k\Sigma_kV_k^T$
2. **Recombine with phase**: $X_k = M_k * exp(j * angle(X))$ (reusing original phase)
3. **Inverse STFT**: Convert back to audio waveform

**How to use:**
1. Drag the slider to your desired k value
2. Click **"🔊 Reconstruct Audio"** to generate the audio
3. Wait for the spectrogram and audio to appear

**Listen for:**
- Low k: Muffled, "underwater" sound - only the dominant frequencies survive
- Medium k: Recognizable but slightly degraded
- High k: Indistinguishable from original


In [ ]:
# Output area for reconstruction results
output_area = widgets.Output()

def reconstruct_audio(k):
    """
    Reconstruct audio using only the top k singular components.
    """
    # Low-rank approximation: M_k = U_k @ Sigma_k @ V_k^T
    Mk = (U[:, :k] * S[:k]) @ Vt[:k, :]

    # Recombine with original phase for audio reconstruction
    Xk = Mk * np.exp(1j * P.astype(np.float32))

    # Inverse STFT to get audio
    yk = librosa.istft(Xk, hop_length=hop_length, length=len(y))
    
    del Xk
    
    # Calculate metrics (use pre-computed cumulative energy)
    energy_captured = cum_energy[k-1] * 100
    
    # Compute error without creating full difference matrix
    diff_norm_sq = np.sum((M - Mk) ** 2)
    m_norm_sq = np.sum(M ** 2)
    frobenius_error = np.sqrt(diff_norm_sq / m_norm_sq) * 100
    
    compression_ratio = (M.shape[0] * M.shape[1]) / (k * (M.shape[0] + M.shape[1] + 1))

    # Clear previous output and display new results
    with output_area:
        clear_output(wait=True)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

        # Original
        librosa.display.specshow(
            librosa.amplitude_to_db(M, ref=np.max),
            sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz',
            cmap=custom_cmap, ax=axes[0]
        )
        axes[0].set_title('Original Spectrogram', fontsize=12, fontweight='bold', color='#e94560')

        # Reconstructed
        librosa.display.specshow(
            librosa.amplitude_to_db(Mk, ref=np.max),
            sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz',
            cmap=custom_cmap, ax=axes[1]
        )
        axes[1].set_title(f'Rank-{k} Reconstruction ({energy_captured:.1f}% energy)', 
                         fontsize=12, fontweight='bold', color='#feca57')

        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Metrics display (condensed)
        print(f"\n{'─'*50}")
        print(f"  k={k}: {energy_captured:.1f}% energy | {frobenius_error:.1f}% error | {compression_ratio:.0f}x compression")
        print(f"{'─'*50}")

        # Audio playback
        print(f"\n▶️ Reconstructed Audio:")
        display(Audio(yk, rate=sr))
    
    del Mk, yk
    plt.close('all')
    gc.collect()

def on_reconstruct_click(b):
    """Button callback to trigger reconstruction with loading indicator."""
    # Show loading state
    b.disabled = True
    b.description = '⏳ Loading...'
    b.button_style = 'warning'
    
    # Show loading message in output area
    with output_area:
        clear_output(wait=True)
        print("🔄 Reconstructing audio... please wait...")
    
    try:
        # Do the reconstruction
        reconstruct_audio(slider.value)
    finally:
        # Restore button state
        b.disabled = False
        b.description = '🔊 Reconstruct Audio'
        b.button_style = 'danger'

# Create interactive slider (no auto-update)
max_k = min(K_MAX, len(S))  # Cap at computed components
slider = widgets.IntSlider(
    min=1,
    max=max_k,
    step=1,
    value=min(k_90, max_k),  # Start at 90% energy
    description='k =',
    style={'description_width': '40px'},
    layout=Layout(width='70%')
)

# Create reconstruct button
reconstruct_button = widgets.Button(
    description='🔊 Reconstruct Audio',
    button_style='danger',  # Red button
    tooltip='Click to reconstruct audio with selected k value',
    layout=Layout(width='200px', height='40px')
)
reconstruct_button.on_click(on_reconstruct_click)

# Layout: slider and button side by side
controls = widgets.HBox([slider, reconstruct_button], layout=Layout(align_items='center', gap='20px'))

print("🎛️ Adjust k with the slider, then click the button to reconstruct:\n")
display(controls)
display(output_area)

# Show initial reconstruction
reconstruct_audio(min(k_90, max_k))


---

## 8. Reconstruction Error Analysis

Let's quantitatively analyze how reconstruction error decreases as we add more components.

The Frobenius norm error equals the square root of the sum of discarded singular values squared. This beautiful result from the Eckart-Young theorem means the error is exactly determined by what we throw away!


In [ ]:
# Compute reconstruction error for different k values
k_values = np.arange(1, min(len(S)+1, 151))  # Cap at available components
errors = []
total_energy = np.sum(S**2)

for k in k_values:
    # Error = sqrt(sum of remaining singular values squared)
    remaining_energy = np.sum(S[k:]**2)
    relative_error = np.sqrt(remaining_energy / total_energy) * 100
    errors.append(relative_error)

errors = np.array(errors, dtype=np.float32)  # Use float32

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1 = axes[0]
ax1.plot(k_values, errors, color='#e94560', linewidth=2)
ax1.axhline(y=10, color='#feca57', linestyle='--', alpha=0.7, label='10% error')
ax1.axhline(y=5, color='#ff6b6b', linestyle='--', alpha=0.7, label='5% error')
ax1.axhline(y=1, color='#ffffff', linestyle='--', alpha=0.5, label='1% error')
ax1.set_xlabel('Number of Components (k)', fontsize=11)
ax1.set_ylabel('Relative Frobenius Error (%)', fontsize=11)
ax1.set_title('Reconstruction Error vs Components', fontsize=14, fontweight='bold', color='#e94560')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, max(k_values)])

# Log scale
ax2 = axes[1]
ax2.semilogy(k_values, errors, color='#e94560', linewidth=2)
ax2.axhline(y=10, color='#feca57', linestyle='--', alpha=0.7, label='10% error')
ax2.axhline(y=5, color='#ff6b6b', linestyle='--', alpha=0.7, label='5% error')
ax2.axhline(y=1, color='#ffffff', linestyle='--', alpha=0.5, label='1% error')
ax2.set_xlabel('Number of Components (k)', fontsize=11)
ax2.set_ylabel('Relative Frobenius Error (%, log scale)', fontsize=11)
ax2.set_title('Reconstruction Error (Log Scale)', fontsize=14, fontweight='bold', color='#e94560')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, max(k_values)])

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print(f"\nError Analysis:")
print(f"   k = {k_90}: {errors[k_90-1]:.2f}% error (90% energy)")
print(f"   k = {k_95}: {errors[k_95-1]:.2f}% error (95% energy)")
print(f"   k = {k_99}: {errors[min(k_99-1, len(errors)-1)]:.2f}% error (99% energy)")


---

## 9. Quick Comparison: Key Reconstruction Points

Let's directly compare the audio at strategically chosen k values to hear the difference compression makes.


In [ ]:
def quick_reconstruct(k):
    """Reconstruct audio with top k components."""
    Mk = (U[:, :k] * S[:k]) @ Vt[:k, :]
    Xk = Mk * np.exp(1j * P)
    result = librosa.istft(Xk, hop_length=hop_length, length=len(y))
    del Mk, Xk
    gc.collect()
    return result

# Key reconstruction points
key_points = [
    (1, "Rank-1 (dominant pattern)"),
    (k_90, f"Rank-{k_90} (90% energy)"),
    (k_99 if k_99 <= len(S) else len(S), f"Rank-{min(k_99, len(S))} (99% energy)")
]

print("🎵 Audio Comparison at Key Points\n")
print("="*50)

print("\n▶️ Original Audio:")
display(Audio(y, rate=sr))
print()

for k, description in key_points:
    if k <= len(S):
        y_recon = quick_reconstruct(k)
        energy_pct = cum_energy[min(k-1, len(cum_energy)-1)] * 100
        print(f"▶️ {description} - Energy: {energy_pct:.1f}%")
        display(Audio(y_recon, rate=sr))
        del y_recon
        gc.collect()
        print()


---

## 10. Bonus: Listen to the Residual

The residual represents the fine details that don't fit the low-rank model. Let's hear what gets discarded!


In [ ]:
def play_residual(k):
    """Reconstruct and play the residual (what was removed)."""
    # Low-rank approximation
    Mk = (U[:, :k] * S[:k]) @ Vt[:k, :]

    # Residual magnitude
    R = np.maximum(M - Mk, 0).astype(np.float32)
    del Mk

    # Recombine residual with original phase
    Xr = R * np.exp(1j * P)
    del R
    yr = librosa.istft(Xr, hop_length=hop_length, length=len(y))
    del Xr

    # Normalize for playback
    yr = yr / (np.max(np.abs(yr)) + 1e-9)

    energy_removed = (1 - cum_energy[min(k-1, len(cum_energy)-1)]) * 100
    print(f"▶️ Residual (what rank-{k} discarded) - {energy_removed:.1f}% of energy:")
    display(Audio(yr, rate=sr))
    del yr
    gc.collect()

print("🔊 What does SVD discard? Listen to the residuals:\n")

# Play residual for k=1 (everything except the dominant pattern)
print("── Rank-1 keeps only the single most dominant pattern ──")
play_residual(1)
print()

# Play the residual at 90% energy threshold
print(f"── Rank-{k_90} keeps 90% of the energy ──")
if k_90 <= len(S):
    play_residual(k_90)
    print()


---

## 11. PCA Perspective: Same Math, Different Story

### Wait — Isn't PCA Just SVD?

**Yes, essentially!** PCA is SVD applied to **centered** data. The eigenvalues of the covariance matrix are the squared singular values, and the principal components are the right singular vectors of the centered matrix.

So why bother with a separate section?

**Because the framing changes everything about interpretation.**

### The Shift in Perspective

When we did SVD on the spectrogram $M$, we asked:
> *"What are the fundamental building blocks (rank-1 patterns) that combine to form this matrix?"*

Now we'll treat the data differently — each **time frame as a sample** and each **frequency bin as a feature**. This lets us ask:
> *"What does the 'typical' moment of this audio sound like, and how does each moment deviate from that?"*

This reframing gives us something SVD didn't explicitly provide: **a baseline**.

### The Eigenfaces Analogy

This is exactly like **eigenfaces** in computer vision:
- The **mean face** captures what a "typical" face looks like
- Each **eigenface** captures a direction of variation (nose size, lighting, expression...)
- Any face ≈ mean face + weighted sum of eigenfaces

For audio, we get **eigenspectra**:
- The **mean spectrum** $\mu$ captures the "average timbre" of the audio — what the typical moment sounds like
- Each **principal component** $w_i$ is a pattern of frequencies that tend to vary together
- The **PC scores** $z_i(t)$ tell us: "at time $t$, how much is pattern $i$ present?"

$$\text{Spectrum at time } t \approx \mu + \sum_{i=1}^k z_i(t) \cdot w_i$$

### Why This Matters

The mean spectrum is musically meaningful — it's like asking "what key/timbre dominates this piece?" The PCs then tell you "in what ways does it deviate from that baseline?"

Let's hear the difference!


In [ ]:
from sklearn.decomposition import PCA

# ═══════════════════════════════════════════════════════════════════════════
# PCA on Log-Spectrogram
# ═══════════════════════════════════════════════════════════════════════════
# Use log magnitude (more perceptually meaningful, better behaved for PCA)
eps = 1e-6
X_log = np.log(M + eps).T.astype(np.float32)  # Shape: (T, F) - time frames as samples

print(f"📊 PCA Dataset:")
print(f"   • Shape: {X_log.shape[0]} samples (time frames) × {X_log.shape[1]} features (frequency bins)")

# Fit PCA with reasonable number of components
PCA_K_MAX = min(100, X_log.shape[1])  # Cap for memory efficiency
print(f"\n🔢 Fitting PCA with up to {PCA_K_MAX} components...", end=" ")

pca = PCA(n_components=PCA_K_MAX)
Z = pca.fit_transform(X_log)  # Shape: (T, PCA_K_MAX) - PC scores
print("Done!")

# Explained variance
pca_cum_var = np.cumsum(pca.explained_variance_ratio_)
pca_k90 = np.searchsorted(pca_cum_var, 0.90) + 1
pca_k95 = np.searchsorted(pca_cum_var, 0.95) + 1
pca_k99 = min(np.searchsorted(pca_cum_var, 0.99) + 1, PCA_K_MAX)

print(f"\n📈 PCA Variance Thresholds:")
print(f"   • 90% variance: k = {pca_k90} components")
print(f"   • 95% variance: k = {pca_k95} components")
print(f"   • 99% variance: k = {pca_k99} components")

# Plot cumulative variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Individual variance
ax1 = axes[0]
ax1.bar(range(1, min(31, PCA_K_MAX+1)), pca.explained_variance_ratio_[:30] * 100, 
        color='#e94560', alpha=0.8, edgecolor='#ff6b6b')
ax1.set_xlabel('Principal Component', fontsize=11)
ax1.set_ylabel('Variance Explained (%)', fontsize=11)
ax1.set_title('Variance per Component (first 30)', fontsize=12, fontweight='bold', color='#e94560')
ax1.grid(True, alpha=0.3, axis='y')

# Right: Cumulative variance
ax2 = axes[1]
ax2.plot(range(1, PCA_K_MAX+1), pca_cum_var * 100, color='#e94560', linewidth=2)
ax2.axhline(y=90, color='#feca57', linestyle='--', alpha=0.7, label='90%')
ax2.axhline(y=95, color='#ff6b6b', linestyle='--', alpha=0.7, label='95%')
ax2.axhline(y=99, color='#ffffff', linestyle='--', alpha=0.5, label='99%')
ax2.scatter([pca_k90], [90], color='#feca57', s=80, zorder=5, edgecolor='white')
ax2.scatter([pca_k95], [95], color='#ff6b6b', s=80, zorder=5, edgecolor='white')
ax2.annotate(f'k={pca_k90}', (pca_k90, 90), textcoords="offset points", xytext=(8,-12), color='#feca57')
ax2.annotate(f'k={pca_k95}', (pca_k95, 95), textcoords="offset points", xytext=(8,-12), color='#ff6b6b')
ax2.set_xlabel('Number of Components (k)', fontsize=11)
ax2.set_ylabel('Cumulative Variance (%)', fontsize=11)
ax2.set_title('Cumulative Variance Explained', fontsize=12, fontweight='bold', color='#e94560')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, PCA_K_MAX])
ax2.set_ylim([0, 102])

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print("\n💡 Compare with SVD: PCA often needs similar k for same variance, but the")
print("   interpretation differs — PCA captures deviations from the MEAN spectrum.")


### Listen to Individual Principal Components

Just like with SVD, let's hear what each principal component sounds like!

- **Mean spectrum** = the "average" sound (like the mean face in eigenfaces)
- **PC scores × PC loadings** = deviations from the mean

For each PC, we'll play:
1. **The mean spectrum alone** (what the "average" sounds like)
2. **Mean + that PC** (the average plus one direction of variation)


In [ ]:
# Listen to the Mean Spectrum and Individual PCs
print("🎧 Listen to PCA Components")
print("="*60)

# First: The mean spectrum alone (the "average" sound)
print("\n" + "─"*60)
print("📊 MEAN SPECTRUM (the 'average' timbre)")
print("─"*60)
print("This is what the audio sounds like if we only use the mean —")
print("no variation, just the average frequency profile held constant.\n")

# Reconstruct using only the mean (k=0 equivalent)
mean_only_log = np.tile(pca.mean_, (Z.shape[0], 1))  # (T, F)
mean_only_mag = np.maximum(np.exp(mean_only_log).T - eps, 0.0).astype(np.float32)
mean_only_complex = mean_only_mag * np.exp(1j * P)
y_mean = librosa.istft(mean_only_complex, hop_length=hop_length, length=len(y))
y_mean = y_mean / (np.max(np.abs(y_mean)) + 1e-9)
del mean_only_log, mean_only_mag, mean_only_complex

print("▶️ Mean spectrum only (no PCs):")
display(Audio(y_mean, rate=sr))
del y_mean
gc.collect()

# Now: Mean + individual PCs
n_pcs_to_play = 3
print(f"\n{'─'*60}")
print(f"📊 MEAN + INDIVIDUAL PCs (hearing each direction of variation)")
print("─"*60)

for i in range(n_pcs_to_play):
    var_pct = pca.explained_variance_ratio_[i] * 100
    print(f"\n🎵 PC{i+1}: {var_pct:.1f}% of variance")
    
    # Reconstruct with mean + just this one PC
    # Use the actual scores to weight the component
    Xk_log = pca.mean_ + Z[:, i:i+1] @ pca.components_[i:i+1, :]  # (T, F)
    Mk_pc = np.maximum(np.exp(Xk_log).T - eps, 0.0).astype(np.float32)
    del Xk_log
    Xk_complex = Mk_pc * np.exp(1j * P)
    yk_pc = librosa.istft(Xk_complex, hop_length=hop_length, length=len(y))
    yk_pc = yk_pc / (np.max(np.abs(yk_pc)) + 1e-9)
    del Xk_complex, Mk_pc
    
    print(f"▶️ Mean + PC{i+1}:")
    display(Audio(yk_pc, rate=sr))
    del yk_pc
    gc.collect()

print("\n💡 The mean captures the 'baseline' sound. Each PC adds a")
print("   different type of variation — like eigenfaces adding features!")


In [ ]:
# Visualize the Mean Spectrum and Top Principal Components
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
times = librosa.times_like(M, sr=sr, hop_length=hop_length)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Top-left: Mean spectrum (the "average" sound)
ax1 = axes[0, 0]
mean_spectrum = pca.mean_
ax1.plot(freqs, mean_spectrum, color='#feca57', linewidth=2)
ax1.fill_between(freqs, mean_spectrum, alpha=0.3, color='#feca57')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Log Magnitude')
ax1.set_title('Mean Spectrum μ (Average Timbre)', fontsize=12, fontweight='bold', color='#feca57')
ax1.set_xlim([0, sr/2])
ax1.grid(True, alpha=0.3)

# Top-right: First 3 PC loadings (frequency patterns)
ax2 = axes[0, 1]
colors_pc = ['#e94560', '#ff6b6b', '#ffffff']
for i in range(3):
    ax2.plot(freqs, pca.components_[i], color=colors_pc[i], linewidth=1.5, 
             label=f'PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)', alpha=0.9)
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Loading')
ax2.set_title('PC Loadings (Frequency Patterns)', fontsize=12, fontweight='bold', color='#e94560')
ax2.set_xlim([0, sr/2])
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.5)

# Bottom-left: PC1 scores over time
ax3 = axes[1, 0]
ax3.plot(times, Z[:, 0], color='#e94560', linewidth=1, alpha=0.8)
ax3.fill_between(times, Z[:, 0], alpha=0.2, color='#e94560')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('PC1 Score')
ax3.set_title('PC1 Scores Over Time (When PC1 is Active)', fontsize=12, fontweight='bold', color='#e94560')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=0, color='gray', linestyle='-', alpha=0.5)

# Bottom-right: PC2 scores over time
ax4 = axes[1, 1]
ax4.plot(times, Z[:, 1], color='#ff6b6b', linewidth=1, alpha=0.8)
ax4.fill_between(times, Z[:, 1], alpha=0.2, color='#ff6b6b')
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('PC2 Score')
ax4.set_title('PC2 Scores Over Time (When PC2 is Active)', fontsize=12, fontweight='bold', color='#ff6b6b')
ax4.grid(True, alpha=0.3)
ax4.axhline(y=0, color='gray', linestyle='-', alpha=0.5)

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

print("\n💡 The MEAN spectrum is like the 'eigenface mean' — the average sound.")
print("   PC loadings show which frequencies vary together (positive = increase together).")
print("   PC scores show WHEN each pattern is active (high score = pattern is present).")


### Interactive PCA Reconstruction

Just like with SVD, let's reconstruct audio using only the top k principal components!

**The reconstruction formula (exactly like eigenfaces):**
$$\hat{X}_k = \mu + Z_k W_k^\top$$

Where:
- $\mu$ = mean spectrum (the "average")
- $Z_k$ = first k PC scores (how much of each pattern)
- $W_k$ = first k PC loadings (the patterns themselves)


In [ ]:
# PCA Reconstruction with Interactive Slider
pca_output_area = widgets.Output()

def pca_reconstruct_audio(k):
    """Reconstruct audio from k principal components."""
    # Reconstruct log-spectrogram: mean + scores @ components
    Xk_log = pca.mean_ + Z[:, :k] @ pca.components_[:k, :]  # Shape: (T, F)
    
    # Convert back to magnitude spectrogram
    Mk_pca = np.exp(Xk_log).T - eps  # Shape: (F, T)
    Mk_pca = np.maximum(Mk_pca, 0.0).astype(np.float32)  # Ensure non-negative
    del Xk_log
    
    # Recombine with original phase
    Xk_complex = Mk_pca * np.exp(1j * P)
    yk = librosa.istft(Xk_complex, hop_length=hop_length, length=len(y))
    del Xk_complex
    yk = yk / (np.max(np.abs(yk)) + 1e-9)
    
    # Metrics
    variance_captured = pca_cum_var[k-1] * 100
    
    with pca_output_area:
        clear_output(wait=True)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        
        # Original
        librosa.display.specshow(
            librosa.amplitude_to_db(M, ref=np.max),
            sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz',
            cmap=custom_cmap, ax=axes[0]
        )
        axes[0].set_title('Original Spectrogram', fontsize=12, fontweight='bold', color='#e94560')
        
        # PCA Reconstructed
        librosa.display.specshow(
            librosa.amplitude_to_db(Mk_pca, ref=np.max),
            sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz',
            cmap=custom_cmap, ax=axes[1]
        )
        axes[1].set_title(f'PCA Reconstruction (k={k}, {variance_captured:.1f}% var)', 
                         fontsize=12, fontweight='bold', color='#feca57')
        
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        
        print(f"\n{'─'*50}")
        print(f"  PCA k={k}: {variance_captured:.1f}% variance captured")
        print(f"{'─'*50}")
        print(f"\n▶️ PCA Reconstructed Audio:")
        display(Audio(yk, rate=sr))
    
    del Mk_pca, yk
    plt.close('all')
    gc.collect()

def on_pca_reconstruct_click(b):
    """Button callback with loading indicator."""
    b.disabled = True
    b.description = '⏳ Loading...'
    b.button_style = 'warning'
    
    with pca_output_area:
        clear_output(wait=True)
        print("🔄 Reconstructing with PCA... please wait...")
    
    try:
        pca_reconstruct_audio(pca_slider.value)
    finally:
        b.disabled = False
        b.description = '🔊 Reconstruct (PCA)'
        b.button_style = 'success'

# Create PCA slider
pca_slider = widgets.IntSlider(
    min=1,
    max=PCA_K_MAX,
    step=1,
    value=pca_k90,
    description='k =',
    style={'description_width': '40px'},
    layout=Layout(width='70%')
)

# Create PCA button (green to distinguish from SVD)
pca_button = widgets.Button(
    description='🔊 Reconstruct (PCA)',
    button_style='success',  # Green button
    tooltip='Click to reconstruct audio with k principal components',
    layout=Layout(width='200px', height='40px')
)
pca_button.on_click(on_pca_reconstruct_click)

pca_controls = widgets.HBox([pca_slider, pca_button], layout=Layout(align_items='center', gap='20px'))

print("🎛️ PCA Reconstruction — adjust k and click to reconstruct:\n")
display(pca_controls)
display(pca_output_area)

# Show initial reconstruction
pca_reconstruct_audio(pca_k90)


### PCA Residuals: What Gets Discarded?

Just like with SVD, let's listen to what PCA throws away when we use only k components.


In [ ]:
def play_pca_residual(k):
    """Play what PCA discards when using k components."""
    # Full reconstruction with k PCs
    Xk_log = pca.mean_ + Z[:, :k] @ pca.components_[:k, :]
    Mk_pca = np.exp(Xk_log).T - eps
    del Xk_log
    
    # Original log-spectrogram
    M_log_original = np.log(M + eps)
    
    # Residual in log domain, then convert
    residual_log = M_log_original - np.log(np.maximum(Mk_pca, eps))
    del Mk_pca
    
    # Convert residual to magnitude (this is approximate but audible)
    residual_mag = np.maximum(np.exp(np.abs(residual_log)) - 1, 0).astype(np.float32)
    del residual_log
    
    # Reconstruct audio from residual
    Xr = residual_mag * np.exp(1j * P)
    yr = librosa.istft(Xr, hop_length=hop_length, length=len(y))
    yr = yr / (np.max(np.abs(yr)) + 1e-9)
    del Xr, residual_mag
    
    var_kept = pca_cum_var[min(k-1, len(pca_cum_var)-1)] * 100
    var_discarded = 100 - var_kept
    
    print(f"▶️ PCA Residual (k={k}) — {var_discarded:.1f}% variance discarded:")
    display(Audio(yr, rate=sr))
    del yr
    gc.collect()

print("🔊 What does PCA discard? Listen to the residuals:\n")

# Residual for k=1 (everything except mean + PC1)
print("── k=1: Only mean + PC1 kept ──")
play_pca_residual(1)
print()

# Residual at 90% variance threshold
print(f"── k={pca_k90}: 90% variance kept ──")
play_pca_residual(pca_k90)
print()

print("💡 Compare with SVD residuals! PCA residuals represent variance")
print("   around the mean that wasn't captured by the top k PCs.")


### Direct Comparison: SVD vs PCA Reconstruction

Let's hear both methods side-by-side at the same number of components. This reveals the subtle differences in what each method preserves!


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FINAL SHOWDOWN: SVD vs PCA Audio Reconstruction Comparison
# ═══════════════════════════════════════════════════════════════════════════

def compare_svd_pca(k, show_header=True):
    """Compare SVD and PCA reconstructions at the same k."""
    if show_header:
        print(f"\n{'━'*60}")
        print(f"  🎯 k = {k} components")
        print(f"{'━'*60}")
    
    # SVD reconstruction
    Mk_svd = (U[:, :k] * S[:k]) @ Vt[:k, :]
    Xk_svd = Mk_svd * np.exp(1j * P)
    yk_svd = librosa.istft(Xk_svd, hop_length=hop_length, length=len(y))
    yk_svd = yk_svd / (np.max(np.abs(yk_svd)) + 1e-9)
    del Xk_svd, Mk_svd
    
    # PCA reconstruction
    Xk_log = pca.mean_ + Z[:, :k] @ pca.components_[:k, :]
    Mk_pca = np.maximum(np.exp(Xk_log).T - eps, 0.0).astype(np.float32)
    del Xk_log
    Xk_pca = Mk_pca * np.exp(1j * P)
    yk_pca = librosa.istft(Xk_pca, hop_length=hop_length, length=len(y))
    yk_pca = yk_pca / (np.max(np.abs(yk_pca)) + 1e-9)
    del Xk_pca, Mk_pca
    
    # Metrics
    svd_energy = cum_energy[min(k-1, len(cum_energy)-1)] * 100
    pca_var = pca_cum_var[min(k-1, len(pca_cum_var)-1)] * 100
    
    print(f"\n🔴 SVD ({svd_energy:.1f}% energy):")
    display(Audio(yk_svd, rate=sr))
    
    print(f"🟢 PCA ({pca_var:.1f}% variance):")
    display(Audio(yk_pca, rate=sr))
    
    del yk_svd, yk_pca
    gc.collect()

# ═══════════════════════════════════════════════════════════════════════════
print("🎧 FINAL COMPARISON: SVD vs PCA Reconstruction")
print("="*60)
print("\nListen to how each method reconstructs the audio at different k values.")
print("Can you hear the difference between SVD and PCA?\n")

print("━"*60)
print("  📌 ORIGINAL AUDIO (Reference)")
print("━"*60)
display(Audio(y, rate=sr))

# Compare at multiple k values
compare_svd_pca(1)   # Extreme compression - just 1 component
compare_svd_pca(5)   # Low k
compare_svd_pca(min(max(k_90, pca_k90), K_MAX, PCA_K_MAX))  # 90% threshold

print(f"\n{'═'*60}")
print("  🎓 WHAT TO LISTEN FOR")
print("═"*60)
print("""
• At k=1: Both are heavily compressed. SVD captures the single 
  most dominant pattern; PCA captures the mean + top deviation.

• At k=5: Still lossy but more recognizable. Notice how SVD and 
  PCA preserve different aspects of the sound.

• At 90% threshold: Nearly indistinguishable from original for 
  both methods, but subtle differences may exist.

Key insight: SVD finds the best low-rank matrix approximation,
while PCA finds maximum variance directions around the mean.
Same math (both use eigendecomposition), different framing!
""")


---

## 12. Summary and Key Takeaways

### What We Demonstrated:

1. **Audio → Matrix**: The spectrogram is a natural matrix representation of sound (frequency × time)

2. **SVD Decomposition**: $M = U \Sigma V^\top$ reveals underlying "audio atoms"
   - Each singular vector pair captures a time-frequency pattern
   - Singular values indicate importance
   - Reconstruction: sum of rank-1 outer products

3. **PCA Decomposition**: Same eigendecomposition math, different framing
   - Treating time frames as samples gives us a **baseline** (mean spectrum)
   - Principal components = directions of maximum variation from that baseline
   - Reconstruction: mean + weighted eigenspectra (just like eigenfaces!)

4. **SVD vs PCA — Same Math, Different Story**:
   
   The core math is essentially identical — PCA is SVD on centered data. The difference is **what question you're asking**:
   
   | SVD asks | PCA asks |
   |----------|----------|
   | "What rank-1 patterns combine to form this matrix?" | "What's typical, and how does each moment deviate?" |
   | No baseline — builds from zero | Explicit baseline (the mean spectrum) |
   | Reconstruction: sum of atoms | Reconstruction: mean + deviations |

5. **Compression**: Both methods capture 90%+ of information with a tiny fraction of components

6. **Perceptual Quality**: The human ear is surprisingly tolerant of lossy compression!

### The Mathematical Beauty:

**SVD**: $M \approx \sum_{i=1}^k \sigma_i u_i v_i^\top$ — adding "audio atoms" one by one

**PCA**: $X_t \approx \mu + \sum_{i=1}^k z_i(t) w_i$ — the mean plus weighted deviations (exactly like eigenfaces!)

Both reveal that complex signals have low intrinsic dimensionality — most of the "information" lives in a small subspace.

---

### Try Different Audio Types!

- **Speech**: Compresses well (structured formants, limited pitch range)
- **Music**: Varies by genre (simple melodies vs complex orchestration)
- **Noise**: Doesn't compress well (energy spread across all components)

---
